<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/08-operations/03-experiment-tracking-and-registry.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 实验追踪与模型注册表

**目标：** 端到端用一遍 MLflow。把一次评测运行的 params/metrics/artifacts 记录下来，比较多次运行来做决策，然后把胜出的配置注册并做阶段性晋升。这套工具能把「我曾经跑过一次评测」变成一个被追踪、可复现、可晋升的工作流。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

## 这个 notebook 为什么存在

第 04 节搭了一个评测 harness：跑一个配置，得到一个数字。第 08/01 节追踪了生产调用。但在「我在 notebook 里得到了一个数字」和「我们团队能安全地交付模型变更」之间，横着一道鸿沟：**这些数字存在哪里，你怎么拿上周的跑跟今天的比，以及生产上跑的确切是哪个配置？**

那就是实验追踪加一个模型注册表，而 **MLflow** 是这方面业界标准的开源工具。这是本仓库里第一个伸手去够真实框架、而不是裸 API 的 notebook，而且是刻意的，因为追踪属于*基础设施*，不是你在生产里会手搓的模式。（对比第 04 节，它从零教你评测的*逻辑*，好让你理解被追踪的到底是什么。）

> **⭐ 关键收获 ——** MLflow 不运行你的评测——它*记住*它们。每次运行的 params、metrics 和 artifacts 都变成可查询的历史，而注册表把「哪个版本在线上？」从部落知识变成一次查表。正是这份记忆，让模型变更成为一个*决策*，而不是一次猜测。

## 准备工作

一如既往地自包含。跟本仓库其余部分有两点不同：

1. 这个 notebook 会安装 **MLflow**（它要教的工具），而不只是 `aien`。
2. 它**不需要 API key**。为了把焦点留在 MLflow 上，「被测系统」是一个小的确定性打分器（第 02 节黄金集那个想法）。真正会发生模型调用的那个单元格被标了出来，好让你清楚看到第 04 节的 harness 该从哪儿插进去。

In [ ]:
%pip install -q mlflow "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
import mlflow

# MLflow 默认把日志写到本地的 ./mlruns 目录——不需要服务器，也不需要账号。
# 在团队里你会把 tracking_uri 指向一个共享的 MLflow 服务器；代码一模一样。
mlflow.set_experiment("rag-eval")   # 把相关的运行归组；首次使用时创建
print("MLflow", mlflow.__version__, "-> logging to ./mlruns")

## 被测系统（你第 04 节 harness 的替身）

为了不靠 key 也能跑，这个「RAG 系统」是一个在极小黄金集上跑的确定性打分器：每个配置产出一个质量分和一个成本。在你的真实项目里，**这个函数就是你第 04 节的 `run_eval`**。它会调用模型、拿黄金集打分，并返回同样的形状。MLflow 不在乎是哪个；你交给它什么数字，它就追踪什么数字。

In [ ]:
# 一个极小的黄金集（第 02/04 节的想法）：问题配上已知的正确答案。
GOLDEN = [
    {"q": "What port does the protocol use?", "key": "8080"},
    {"q": "Who ratified the spec?", "key": "the working group"},
    {"q": "What is the max message size?", "key": "64 KB"},
]

def evaluate(config):
    """第 04 节 run_eval 的替身。返回 MLflow 将要追踪的 metrics。
    真实版本：用 `config` 调用模型，拿输出对 GOLDEN 打分。
    """
    # 确定性的玩具打分：更多 chunk + 一个 reranker -> 更高准确率、更高成本。
    base = 0.55 + 0.08 * config["top_k"] + (0.12 if config["rerank"] else 0.0)
    accuracy = min(base, 0.98)
    cost_per_query = 0.0002 * config["top_k"] * (2.0 if config["rerank"] else 1.0)
    latency_ms = 120 + 40 * config["top_k"] + (200 if config["rerank"] else 0)
    return {"accuracy": round(accuracy, 3),
            "cost_per_query": round(cost_per_query, 5),
            "latency_ms": latency_ms}

print(evaluate({"top_k": 3, "rerank": False}))

## 记录一次运行：params 进，metrics + artifacts 出

MLflow 的核心循环就是一个上下文管理器：打开一次 run，记录 **params**（配置），记录 **metrics**（结果），记录任何 **artifacts**（文件：评测报告、一张图、黄金集）。`with mlflow.start_run()` 里面的一切都被作为一个可比对的记录一起捕获。

In [ ]:
import json, tempfile, os

def tracked_eval(config, run_name):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(config)                 # 你拧的那些旋钮
        metrics = evaluate(config)
        mlflow.log_metrics(metrics)               # 你拿到的那些数字

        # 把完整结果作为 artifact 记录——评测「报告」随这次运行一起走。
        report = {"config": config, "metrics": metrics, "golden_set_size": len(GOLDEN)}
        path = os.path.join(tempfile.mkdtemp(), "eval_report.json")
        with open(path, "w") as f:
            json.dump(report, f, indent=2)
        mlflow.log_artifact(path)
        return metrics, run.info.run_id

m, _ = tracked_eval({"top_k": 3, "rerank": False}, run_name="baseline")
print("logged baseline:", m)

## 扫一遍配置 —— 这才是追踪的*用武之地*

一次运行只是一行日志。真正的回报是许多次可比较的运行。下面我们扫一遍 RAG 的那些旋钮，每一次都变成一次被追踪的运行，参数和指标并排放着。这正好就是第 04 节那个「交付一个变更、度量它」的循环，现在带上了记忆。

In [ ]:
sweep = [
    {"top_k": 1, "rerank": False},
    {"top_k": 3, "rerank": False},
    {"top_k": 5, "rerank": False},
    {"top_k": 3, "rerank": True},
    {"top_k": 5, "rerank": True},
]
for cfg in sweep:
    name = f"k{cfg['top_k']}{'_rerank' if cfg['rerank'] else ''}"
    tracked_eval(cfg, run_name=name)
print(f"logged {len(sweep)} runs to the 'rag-eval' experiment")

## 用代码比较运行并挑出胜者

MLflow 有一个 UI（本地 `mlflow ui`，或团队里那个 tracking server 的 web app），但一切也可以用代码查询，这就是你在 CI 里给决策设卡的方式。把运行拉回来，按你关心的那个取舍给它们排序。

In [ ]:
runs = mlflow.search_runs(experiment_names=["rag-eval"])
# 展示要紧的那几列，准确率最高的排最前。
cols = ["tags.mlflow.runName", "params.top_k", "params.rerank",
        "metrics.accuracy", "metrics.cost_per_query", "metrics.latency_ms"]
view = runs[cols].sort_values("metrics.accuracy", ascending=False)
print(view.to_string(index=False))

# 真实关卡很少是「最高准确率」——而是成本/延迟预算下最好的准确率。
budget = runs[(runs["metrics.cost_per_query"] <= 0.002) & (runs["metrics.latency_ms"] <= 400)]
winner = budget.sort_values("metrics.accuracy", ascending=False).iloc[0]
print(f"\nwinner under budget: {winner['tags.mlflow.runName']} "
      f"(acc={winner['metrics.accuracy']}, ${winner['metrics.cost_per_query']}/q, "
      f"{int(winner['metrics.latency_ms'])}ms)")

> **🔵 面试信号 ——** 「我们挑的是*在成本和延迟预算下*准确率最好的配置，而且这次运行被记录了下来，任何人都可以复现」是一个生产级的回答。它说明你是对着约束做优化（09/02、10/01），也说明你的选择是可审计的——而不是「它感觉更好」。

## 注册胜者：模型注册表

追踪记住的是*实验*。**注册表**回答的是另一个问题：*哪个版本是我们祝福用于生产的那个？* 你把一个模型注册成一个具名条目，它就得到一个版本号。然后你把一个会移动的 **alias**（比如 `champion`）指向那个被祝福的版本，于是「线上跑的是什么？」就是一次查表，而回滚就是把 alias 重新指向上一个版本，而不是一阵手忙脚乱。

（MLflow 3 用 *alias* 做这件事；更老的教程用 `Staging`/`Production` 这类 *stage*，它们现在已废弃。同一个想法，更新的 API。）

为了注册一个有意义的东西，我们把胜出的配置记录成一个极小的 **pyfunc model**（一个真实、可加载的 artifact），而不只是一个 JSON 文件。在你的项目里，这就是你真实的 RAG pipeline；注册表流程在两种情况下一模一样。

In [ ]:
from mlflow import MlflowClient
from mlflow.pyfunc import PythonModel

MODEL_NAME = "rag-answerer"
client = MlflowClient()

# 上面那轮搜索里胜出的配置。
winner_cfg = {"top_k": int(winner["params.top_k"]), "rerank": winner["params.rerank"] == "True"}

class RagAnswerer(PythonModel):
    """你真实 RAG pipeline 的替身。在生产里 predict() 会做检索 + 生成。"""
    def predict(self, context, model_input):
        return [f"answer (top_k={winner_cfg['top_k']}, rerank={winner_cfg['rerank']})"
                for _ in range(len(model_input))]

# 在一次 run 里记录这个模型，然后把那个被记录的模型注册成一个具名版本。
with mlflow.start_run(run_name="register-winner"):
    mlflow.log_params(winner_cfg)
    mlflow.log_metrics(evaluate(winner_cfg))
    info = mlflow.pyfunc.log_model(name="model", python_model=RagAnswerer())

mv = mlflow.register_model(model_uri=info.model_uri, name=MODEL_NAME)
print(f"registered {MODEL_NAME} version {mv.version}")

# 把 'champion' alias 指向这个版本。以后重新指向它，就是晋升/回滚。
client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)
print(f"alias champion -> {MODEL_NAME} v{mv.version}")

In [ ]:
# 「现在线上跑的是什么？」——按 alias 查表，而不是靠部落知识。
live = client.get_model_version_by_alias(MODEL_NAME, "champion")
run = client.get_run(live.run_id)
print(f"CHAMPION: {MODEL_NAME} v{live.version}")
print(f"  config : top_k={run.data.params['top_k']}, rerank={run.data.params['rerank']}")
print(f"  metrics: accuracy={run.data.metrics['accuracy']}, "
      f"cost/q=${run.data.metrics['cost_per_query']}")
print("  -> rollback = re-point 'champion' at the previous version; audit = read this run's params")

> **⚠️ 生产现实 ——** 注册表有多可信，取决于围绕它的纪律。*只*注册通过了评测关卡（第 04 节）的配置，并把晋升到 `Production` 变成一个经过评审的步骤，而不是自动的。一个塞满未评测的「latest」版本的注册表，只是一种更花哨的失控方式——工具不会创造严谨，它只是*记录*严谨。

## 它在技术栈里的位置

MLflow 闭合了本仓库一直在搭的那个回路：

- **第 02/04 节的评测**产出数字 → **08/03（本篇）** 追踪并比较它们。
- **08/01 可观测性**产出生产指标 → 把漂移信号作为新的被追踪运行喂回来。
- **10/01 系统设计**在预算下选一个配置 → 注册表记录的是*哪个*版本，所以部署的服务和那次评测可证明是同一个东西。

这条可追溯性——从一个评测数字到正在服务流量的确切版本——就是 AI 系统团队期望你知道其存在的那个「MLOps」，哪怕服务器是由某个平台团队在跑的。

## 实践与反模式

| ✅ 该做 | ❌ 反模式 |
|---|---|
| 每次运行都把 params、metrics *以及*评测报告 artifact 一起记录 | 只记一个指标，日后复现不出是哪个配置产出了它 |
| 用代码在成本/延迟预算下挑胜者，让它成为一道关卡 | 对着 UI 瞅一眼就说「这次看起来最好」 |
| 只注册通过了第 04 节评测关卡的配置 | 注册每一个「latest」运行，让信号淹没在噪声里 |
| 用 alias（例如 `champion`）表示线上是哪个；回滚 = 重新指向它 | 把「线上跑的是什么」记在一条 Slack 消息或某人的记忆里 |
| 团队里把 tracking_uri 指向共享服务器；代码保持不变 | 把运行留在别人看不到的本地 ./mlruns 里，还管这叫追踪 |
| 把注册表当作你已有的严谨的记录 | 指望工具创造出你并没有实践的纪律 |

## 练习

1. **接上真实 harness。** 把 `evaluate()` 换成你第 04 节真正的 `run_eval`（它需要一个 Groq key）。确认*整个* MLflow 循环（记录、扫描、比较、注册）原样可用。这种不变性就是要点：你交给 MLflow 什么指标，它就追踪什么。
2. **更严的关卡。** 改一下预算过滤，再要求 `latency_ms <= 300`。注意胜者变了，把它注册成一个新版本，并把 `champion` 指向它。读回这个 alias，确认它现在解析到新版本。
3. **在 UI 里看看它。** 运行 `!mlflow ui`（在本地），或者读一下 `./mlruns` 的目录结构。找出每次运行的 params、metrics 和你的 `eval_report.json` artifact 存在哪里。理解落盘形状能祛魅这个工具。
4. **回滚演练。** 把一个*更差*的配置注册成新版本，把 `champion` 指向它，然后通过把 `champion` 重新指回上一个版本来模拟一次回滚。写出那两行 `set_registered_model_alias` 调用，并用一句话说明为什么 alias 让这件事变成 10 秒的操作，而不是一次重新部署。